# Notebook 4: paper figures
Inputs: Stage 0, Notebook 1, final Notebook 2, Notebook 3 outputs. Run all cells; download `paper_figures.zip`.

In [1]:
# =====================================================================
#  Notebook 4: paper figures for the Results section
#  Attach as inputs: Stage 0 output, Notebook 1 output (background removal),
#  the final Notebook 2 version (holds results/all, single, multi) and
#  the Notebook 3 output (analysis/). CPU is enough; runtime a few minutes.
#  Output: /kaggle/working/paper_figures.zip  (PDF with selectable text + 300-dpi PNG)
# =====================================================================
import os, json, zipfile, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from PIL import Image
warnings.filterwarnings("ignore")

INPUT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path(os.environ.get("BDLW_INPUT", "."))
OUT = Path("/kaggle/working/paper_figures") if Path("/kaggle/working").is_dir() else Path("paper_figures")
OUT.mkdir(parents=True, exist_ok=True)

# ---- Times-style font: Times New Roman if present, otherwise Liberation Serif or STIX (bundled with matplotlib)
have = {f.name for f in font_manager.fontManager.ttflist}
SERIF = [f for f in ["Times New Roman", "Liberation Serif", "STIXGeneral", "DejaVu Serif"] if f in have]
plt.rcParams.update({"font.family": SERIF, "mathtext.fontset": "stix", "pdf.fonttype": 42, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#55555a",
                     "xtick.color": "#404040", "ytick.color": "#404040", "legend.frameon": False,
                     "savefig.bbox": "tight", "savefig.pad_inches": 0.03})
print("Font used:", SERIF[0])


def find_dir(pred):
    for d, sub, files in os.walk(INPUT, followlinks=True):
        p = Path(d)
        if pred(p, files):
            return p
        sub[:] = [s for s in sub if s not in ("standardized", "cutouts", "masks", "weights", "figures")]
    return None


STAGE0 = find_dir(lambda p, f: p.name == "metadata" and "manifest.csv" in f)
STAGE0 = STAGE0.parent if STAGE0 else None
BGR = find_dir(lambda p, f: p.name == "bg_removed" and "mask_manifest.csv" in f)
NB2 = find_dir(lambda p, f: p.name == "results" and (p / "all" / "summary_all.csv").exists())
NB3 = find_dir(lambda p, f: p.name == "analysis" and (p / "tables").is_dir())
print("Stage 0:", STAGE0, "\nNotebook 1:", BGR, "\nNotebook 2:", NB2, "\nNotebook 3:", NB3)

CL = ["Churi", "Dao", "Khor", "Kodal", "Kural", "Shabol"]
SCN = {"all": "E1 All", "single": "E2 Single", "multi": "E3 Multi"}
MODELS = ["resnet50", "efficientnet_b0", "convnext_tiny", "vit_small_patch16_224"]
MOD = {"resnet50": "ResNet-50", "efficientnet_b0": "EfficientNet-B0", "convnext_tiny": "ConvNeXt-Tiny",
       "vit_small_patch16_224": "ViT-S/16"}
MCOL = {"resnet50": "#2F5597", "efficientnet_b0": "#548235", "convnext_tiny": "#7F63B8", "vit_small_patch16_224": "#C55A11"}
BLUES = LinearSegmentedColormap.from_list("b", ["#ffffff", "#DAE3F3", "#8FAADC", "#2F5597", "#1F3864"])
made = []


def save(fig, name):
    fig.savefig(OUT / f"{name}.pdf"); fig.savefig(OUT / f"{name}.png", dpi=300); plt.close(fig)
    made.append(name); print("saved", name)


def best_model(sc):
    t = pd.read_csv(NB3 / "tables" / "T7_scenarios.csv")
    return t.set_index("scenario").loc[sc, "best_model"]


# =============================================================== Fig A: training curves (train loss per epoch, F)
if NB2:
    fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.1), gridspec_kw=dict(wspace=0.32, width_ratios=[1, 1, 1, 0.85]))
    for ax, sc in zip(axes[:3], ["all", "single", "multi"]):
        for m in MODELS:
            f = NB2 / sc / m / "F" / "train_log.csv"
            if not f.exists():
                continue
            d = pd.read_csv(f)
            g = d.groupby("epoch").train_loss
            mu, sd = g.mean(), g.std().fillna(0)
            ax.plot(mu.index, mu.values, color=MCOL[m], lw=1.4, label=MOD[m])
            ax.fill_between(mu.index, mu - sd, mu + sd, color=MCOL[m], alpha=0.15, lw=0)
        ax.set_title(f"({'abc'[list(SCN).index(sc)]}) {SCN[sc]}", loc="left", fontsize=9.5)
        ax.set_xlabel("Epoch"); ax.grid(axis="y", color="#E7E6E6", lw=0.6)
    axes[0].set_ylabel("Training loss")
    # learning-rate schedule (1-epoch warm-up, cosine decay)
    ep = np.linspace(0, 25, 500); lr = np.where(ep < 1, 2e-4 * ep, 1e-4 * (1 + np.cos(np.pi * (ep - 1) / 24)))
    axes[3].plot(ep, lr * 1e4, color="#404040", lw=1.4); axes[3].set_title("(d) Learning rate", loc="left", fontsize=9.5)
    axes[3].set_xlabel("Epoch"); axes[3].set_ylabel(r"LR ($\times10^{-4}$)")
    axes[0].legend(loc="upper center", bbox_to_anchor=(2.1, -0.32), ncol=4, fontsize=8)
    save(fig, "res_training_curves")

    # training time per pipeline (table, ResNet-50 ablation + final models)
    rows = []
    for sc in SCN:
        s = pd.read_csv(NB2 / sc / "summary_all.csv").drop_duplicates(["model", "pipeline"], keep="last")
        for r in s.itertuples():
            rows.append(dict(scenario=sc, model=r.model, pipeline=r.pipeline, minutes=r.minutes,
                             n_trainval=r.n_trainval, n_test=r.n_test))
    pd.DataFrame(rows).to_csv(OUT / "training_time_per_pipeline.csv", index=False)
    made.append("training_time_per_pipeline.csv")

# =============================================================== Fig B: confusion matrices from predictions
if NB2 and NB3:
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.55), gridspec_kw=dict(wspace=0.35))
    for ax, sc, tag in zip(axes, SCN, "abc"):
        m = best_model(sc)
        p = pd.read_csv(NB2 / sc / m / "F" / "test_predictions.csv")
        cm = pd.crosstab(p.y, p.pred).reindex(index=range(6), columns=range(6), fill_value=0).values
        n = cm / cm.sum(1, keepdims=True)
        ax.imshow(n, cmap=BLUES, vmin=0, vmax=1)
        for i in range(6):
            for j in range(6):
                if cm[i, j]:
                    ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7.5,
                            color="white" if n[i, j] > 0.5 else "#1F3864")
        ax.set_xticks(range(6), CL, rotation=45, ha="right", fontsize=7.5); ax.set_yticks(range(6), CL, fontsize=7.5)
        ax.set_title(f"({tag}) {SCN[sc]}, {MOD[m]}", fontsize=9, loc="left"); ax.set_xlabel("Predicted", fontsize=8)
        ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_visible(True); s.set_color("#BFBFBF")
    axes[0].set_ylabel("True", fontsize=8)
    save(fig, "res_confusion")

# =============================================================== Fig C: background kept vs removed (other 3 models)
if NB3:
    three = ["efficientnet_b0", "convnext_tiny", "vit_small_patch16_224"]
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.2), sharey=True, gridspec_kw=dict(wspace=0.08))
    for ax, sc, tag in zip(axes, SCN, "abc"):
        d = pd.read_csv(NB3 / sc / "tables" / "T3_background.csv").set_index("model")
        for i, m in enumerate(three):
            k, r = d.loc[m, "kept"], d.loc[m, "removed"]
            ax.plot([r, k], [i, i], color="#BFBFBF", lw=2.2, zorder=1)
            ax.scatter([k], [i], s=42, color="#2F5597", zorder=3, label="background kept" if i == 0 else None)
            ax.scatter([r], [i], s=42, color="#ED7D31", zorder=3, label="background removed" if i == 0 else None)
            ax.text((k + r) / 2, i + 0.22, f"+{k - r:.3f}", ha="center", fontsize=7.5, color="#404040")
        ax.set_yticks(range(3), [MOD[m] for m in three]); ax.set_xlim(0.86, 1.005); ax.set_ylim(-0.5, 2.6)
        ax.set_title(f"({tag}) {SCN[sc]}", loc="left", fontsize=9.5); ax.set_xlabel("CV macro-F1")
        ax.tick_params(axis="y", length=0)
    axes[0].legend(loc="upper center", bbox_to_anchor=(1.6, -0.32), ncol=2, fontsize=8)
    save(fig, "res_background")

# =============================================================== Fig D: cross-condition heatmap
if NB3:
    d = pd.read_csv(NB3 / "tables" / "T8_cross_condition.csv")
    cols = [("single", "test_single_macro_f1", "Single → Single"), ("single", "test_multi_macro_f1", "Single → Multi"),
            ("multi", "test_multi_macro_f1", "Multi → Multi"), ("multi", "test_single_macro_f1", "Multi → Single")]
    M = np.array([[d[(d.model == m) & (d.trained_on == tr)][c].iloc[0] for tr, c, _ in cols] for m in MODELS])
    fig, ax = plt.subplots(figsize=(4.6, 2.15))
    cmap = LinearSegmentedColormap.from_list("rg", ["#C00000", "#F4B183", "#FFF2CC", "#A9D18E", "#548235"])
    ax.imshow(M, cmap=cmap, vmin=0.5, vmax=1.0, aspect="auto")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{M[i, j]:.3f}", ha="center", va="center", fontsize=8.5)
    ax.set_xticks(range(4), [c[2] for c in cols], fontsize=8); ax.set_yticks(range(4), [MOD[m] for m in MODELS], fontsize=8.5)
    ax.xaxis.tick_top(); ax.tick_params(length=0); ax.axvline(1.5, color="white", lw=3)
    for s in ax.spines.values():
        s.set_visible(False)
    save(fig, "res_cross_condition")

# =============================================================== Fig E: robustness curves
if NB3:
    d = pd.read_csv(NB3 / "tables" / "T11a_robustness_all.csv")
    corr = [("blur", "Gaussian blur"), ("dark", "Darkness"), ("jpeg", "JPEG compression"), ("noise", "Gaussian noise"),
            ("occlusion", "Occlusion"), ("rotation", "Rotation")]
    fig, axes = plt.subplots(2, 3, figsize=(7.2, 3.9), sharex=True, sharey=True, gridspec_kw=dict(hspace=0.35, wspace=0.08))
    for ax, (c, t) in zip(axes.ravel(), corr):
        for m in MODELS:
            s = d[(d.model == m) & (d.corruption.isin([c, "clean"]))].sort_values("severity")
            ax.plot(s.severity, s.macro_f1, marker="o", ms=3.2, lw=1.4, color=MCOL[m], label=MOD[m])
        ax.set_title(t, fontsize=9.5); ax.set_ylim(0.1, 1.03); ax.set_xticks(range(6), ["clean", 1, 2, 3, 4, 5])
        ax.grid(axis="y", color="#E7E6E6", lw=0.6)
    for ax in axes[1]:
        ax.set_xlabel("Severity level")
    for ax in axes[:, 0]:
        ax.set_ylabel("Test macro-F1")
    axes[0, 0].legend(loc="upper center", bbox_to_anchor=(1.6, -1.75), ncol=4, fontsize=8)
    save(fig, "res_robustness")

# =============================================================== Fig F: Grad-CAM (tiles from Notebook 3 figure)
if NB3 and (NB3 / "all" / "figures" / "F6_gradcam.png").exists():
    im = np.asarray(Image.open(NB3 / "all" / "figures" / "F6_gradcam.png").convert("RGB"))
    nw = (im < 240).any(2)

    def bands(p, th, minlen):
        out, s0 = [], None
        for i, v in enumerate(list(p) + [0]):
            if v > th and s0 is None: s0 = i
            if v <= th and s0 is not None:
                if i - s0 > minlen: out.append((s0, i))
                s0 = None
        return out
    rws = bands(nw.sum(1), 300, 100); cls_ = bands(nw[rws[0][0]:rws[0][1]].sum(0), 50, 100)
    show, heads = [0, 1, 2, 4], ["Input", "K1 (kept)", "F (final)", "R1 (removed)"]
    fig, axes = plt.subplots(4, 6, figsize=(7.2, 5.0), gridspec_kw=dict(wspace=0.03, hspace=0.04))
    for j, (y0, y1) in enumerate(rws[:6]):
        for i, c in enumerate(show):
            x0, x1 = cls_[c]
            ax = axes[i, j]; ax.imshow(im[y0:y1, x0:x1]); ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values(): s.set_visible(False)
            if i == 0: ax.set_title(CL[j], fontsize=9.5, fontweight="bold")
            if j == 0: ax.set_ylabel(heads[i], fontsize=8.5)
    save(fig, "res_gradcam")

# =============================================================== Fig G: sample test predictions (clear images)
if NB2 and NB3 and STAGE0:
    man = pd.read_csv(STAGE0 / "metadata" / "manifest.csv", dtype={"image_id": str})
    m = best_model("all")
    p = pd.read_csv(NB2 / "all" / m / "F" / "test_predictions.csv", dtype={"image_id": str})
    p = p.merge(man[["image_id", "path", "lap_var", "orig_w", "orig_h"]], on="image_id", how="left")
    p["conf"] = p[[f"p_{c}" for c in CL]].max(axis=1)
    # Candidate sheet: the 8 sharpest correctly classified test images per class and subset, with their IDs.
    # Look at candidates_*.png, then put the clearest IDs into PICK_IDS below and run this cell again.
    PICK_IDS = ["c4906d290826", "1eca59899e20",   # Churi  single, multi
                "76feb6d0244c", "be1ae96f900d",   # Dao
                "970b66372e0b", "68f68ec7a53a",   # Khor
                "0d57eb7a16c0", "c0306633edfd",   # Kodal
                "6a4709662d97", "b3c062594822",   # Kural
                "31ea9395750f", "e91baaba0b44"]   # Shabol   (empty list = choose automatically)
    for sub in ["single", "multi"]:
        cand = [p[(p.y == c) & (p.subset == sub) & (p.pred == c)].sort_values("lap_var", ascending=False).head(8)
                for c in range(6)]
        fig, axes = plt.subplots(6, 8, figsize=(9, 7.2), gridspec_kw=dict(wspace=0.05, hspace=0.3))
        for c, q in enumerate(cand):
            for k in range(8):
                ax = axes[c, k]; ax.axis("off")
                if k < len(q):
                    r = q.iloc[k]; a = Image.open(STAGE0 / r.path).convert("RGB"); a.thumbnail((200, 200))
                    ax.imshow(a); ax.set_title(r.image_id, fontsize=5.5)
            axes[c, 0].text(-0.1, 0.5, CL[c], transform=axes[c, 0].transAxes, ha="right", va="center", fontsize=8)
        fig.savefig(OUT / f"candidates_{sub}.png", dpi=150); plt.close(fig); made.append(f"candidates_{sub}.png")
    picks = []
    if PICK_IDS:
        picks = [p[p.image_id == i].iloc[0] for i in PICK_IDS if (p.image_id == i).any()]
    else:
        for c in range(6):                              # sharpest correct single and multi image per class
            for sub in ["single", "multi"]:
                q = p[(p.y == c) & (p.subset == sub) & (p.pred == c)].sort_values("lap_var", ascending=False)
                if len(q): picks.append(q.iloc[0])
    wrong = p[p.pred != p.y]                            # every error of the best model is shown
    picks += [r for _, r in wrong.iterrows()]
    n = len(picks); ncol = 6; nrow = math.ceil(n / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(7.2, 1.42 * nrow), gridspec_kw=dict(wspace=0.04, hspace=0.32))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel(): ax.axis("off")
    for ax, r in zip(axes.ravel(), picks):
        a = Image.open(STAGE0 / r.path).convert("RGB")
        w, h = a.size; s = min(w, h); a = a.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((400, 400))
        ok = r.pred == r.y; col = "#548235" if ok else "#C00000"
        ax.imshow(a); ax.add_patch(Rectangle((0, 0), 399, 399, fill=False, ec=col, lw=2.2))
        ax.set_title(f"{CL[int(r.pred)]} {r.conf:.2f}", fontsize=8, color=col, pad=2)
        if not ok:
            ax.text(200, 385, f"true: {CL[int(r.y)]}", ha="center", va="bottom", fontsize=7, color="white",
                    bbox=dict(fc="#C00000", ec="none", pad=1.2))
    save(fig, "res_sample_predictions")

with zipfile.ZipFile(OUT.parent / "paper_figures.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        z.write(f, f.name)
print("\nDone. Download paper_figures.zip from the Output tab. Files:", sorted(made))


Font used: Liberation Serif
Stage 0: /kaggle/input/notebooks/arnicasarker/stage0-single-multi-3parts/stage0 
Notebook 1: /kaggle/input/notebooks/arnicasarker/nb1-background-removal/bg_removed 
Notebook 2: /kaggle/input/notebooks/arnicasarker/nb2-training-scenarios/results 
Notebook 3: /kaggle/input/notebooks/arnicasarker/nb3-analysis-scenarios/analysis
saved res_training_curves
saved res_confusion
saved res_background
saved res_cross_condition
saved res_robustness
saved res_gradcam
saved res_sample_predictions

Done. Download paper_figures.zip from the Output tab. Files: ['candidates_multi.png', 'candidates_single.png', 'res_background', 'res_confusion', 'res_cross_condition', 'res_gradcam', 'res_robustness', 'res_sample_predictions', 'res_training_curves', 'training_time_per_pipeline.csv']


In [2]:
# =====================================================================
#  Notebook 4, cell 2: Grad-CAM figure on chosen clear test images
#  Uses the saved fold-0 weights of Notebook 2 (results/all/<model>/{K1,R1,F}/weights/fold0.pt).
#  Run after cell 1 (it reuses STAGE0, BGR, NB2, CL, OUT, save, PICK_IDS). CPU is enough (about 1 minute).
# =====================================================================
import math, cv2, torch, timm

# one clear test image per class (Churi, Dao, Khor, Kodal, Kural, Shabol); change IDs here if wanted
CAM_IDS = ["c4906d290826", "76feb6d0244c", "970b66372e0b", "0d57eb7a16c0", "6a4709662d97", "31ea9395750f"]
CAM_MODEL = best_model("all")                  # ConvNeXt-Tiny in E1
DEV = "cuda" if torch.cuda.is_available() else "cpu"
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

summ = pd.read_csv(NB2 / "all" / "summary_all.csv").drop_duplicates(["model", "pipeline"], keep="last")
man = pd.read_csv(STAGE0 / "metadata" / "manifest.csv", dtype={"image_id": str})
mm = pd.read_csv(BGR / "mask_manifest.csv", dtype={"image_id": str})
man = man.merge(mm[["image_id", "cutout_path"]], on="image_id", how="left").set_index("image_id")


def cfg_of(pipe):
    s = summ[(summ.model == CAM_MODEL) & (summ.pipeline == pipe)].iloc[0]
    return dict(bg=s.cfg_bg, size=s.cfg_size, res=int(s.cfg_res), color=s.cfg_color)


def geom(img, c):
    if c["size"] == "stretch":
        return cv2.resize(img, (c["res"], c["res"]), interpolation=cv2.INTER_AREA)
    h, w = img.shape[:2]; s = c["res"] / max(h, w); nh, nw = max(1, round(h * s)), max(1, round(w * s))
    small = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    out = np.full((c["res"], c["res"], 3), 128, np.uint8); t, l = (c["res"] - nh) // 2, (c["res"] - nw) // 2
    out[t:t + nh, l:l + nw] = small
    return out


def prep(iid, c):
    r = man.loc[iid]
    src = (BGR / r.cutout_path) if (c["bg"] == "removed" and isinstance(r.cutout_path, str)) else (STAGE0 / r.path)
    rgb = np.asarray(Image.open(src).convert("RGB"))
    if c["color"] == "clahe":
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
        lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
        rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    img = geom(rgb, c)
    x = torch.from_numpy(((img / 255.0 - MEAN) / STD).transpose(2, 0, 1)).float()[None].to(DEV)
    return img, x


def load(pipe):
    c = cfg_of(pipe); kw = dict(pretrained=False, num_classes=6)
    if CAM_MODEL.startswith("vit"): kw["img_size"] = c["res"]
    net = timm.create_model(CAM_MODEL, **kw)
    sd = torch.load(NB2 / "all" / CAM_MODEL / pipe / "weights" / "fold0.pt", map_location="cpu")
    net.load_state_dict({k: v.float() for k, v in sd.items()})
    return net.eval().to(DEV), c


class CAM:
    def __init__(self, net):
        self.net, self.prefix = net, getattr(net, "num_prefix_tokens", 1)
        layer = {"resnet": lambda n: n.layer4, "convnext": lambda n: n.stages[-1], "efficientnet": lambda n: n.bn2,
                 "vit": lambda n: n.blocks[-1].norm1}[[k for k in ("resnet", "convnext", "efficientnet", "vit") if CAM_MODEL.startswith(k)][0]](net)
        layer.register_forward_hook(self.hook)
    def hook(self, m, i, o):
        self.a = o; o.register_hook(lambda g: setattr(self, "g", g))
    def __call__(self, x):
        self.net.zero_grad()
        with torch.enable_grad():
            lo = self.net(x); k = int(lo.argmax(1)); lo[0, k].backward()
        a, g = self.a.detach(), self.g.detach()
        if a.dim() == 3:
            a, g = a[:, self.prefix:], g[:, self.prefix:]; n = int(round(math.sqrt(a.shape[1])))
            a = a.reshape(1, n, n, -1).permute(0, 3, 1, 2); g = g.reshape(1, n, n, -1).permute(0, 3, 1, 2)
        cam = torch.relu((g.mean((2, 3), keepdim=True) * a).sum(1))[0].cpu().numpy()
        cam = cv2.resize(cam, (x.shape[3], x.shape[2])); cam = cam / cam.max() if cam.max() > 0 else cam
        return cam, k, torch.softmax(lo.detach(), 1)[0, k].item()


nets = {p: load(p) for p in ("K1", "F", "R1")}
cams = {p: CAM(nets[p][0]) for p in nets}
overlay = lambda img, cam: (0.55 * img / 255 + 0.45 * plt.get_cmap("inferno")(cam)[..., :3]).clip(0, 1)
heads = ["Input", "K1 (kept)", "F (final)", "R1 (removed)"]
fig, axes = plt.subplots(4, 6, figsize=(7.2, 5.3), gridspec_kw=dict(wspace=0.03, hspace=0.22))
for j, iid in enumerate(CAM_IDS):
    tiles = [(prep(iid, nets["F"][1])[0] / 255, "")]
    for p in ("K1", "F", "R1"):
        img, x = prep(iid, nets[p][1]); cam, k, conf = cams[p](x)
        tiles.append((overlay(img, cam), f"{CL[k]} {conf:.2f}"))
    for i, (im, t) in enumerate(tiles):
        ax = axes[i, j]; ax.imshow(im); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        if t: ax.set_xlabel(t, fontsize=7, labelpad=1)
        if i == 0: ax.set_title(CL[j], fontsize=9.5, fontweight="bold")
        if j == 0: ax.set_ylabel(heads[i], fontsize=8.5)
save(fig, "res_gradcam")

with zipfile.ZipFile(OUT.parent / "paper_figures.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        z.write(f, f.name)
print("Grad-CAM done; paper_figures.zip updated.")


saved res_gradcam
Grad-CAM done; paper_figures.zip updated.
